# DS5 — Information Theory (InT) Model

**Stage 4, Step 1 — Theoretical and Conceptual Modeling**
**Dataset:** DS5 (German Adolescents RCT, N=573)
**Model family:** Information Theory (InT)
**Paper section:** §4.1.5, §4.2.5
**Date:** 2026-04-17
**StoP prerequisite:** `notebooks/43_ds5_stochastic_processes.ipynb` (must be run and outputs saved first)

---

## Scope and purpose

- **Goal:** Estimate per-participant behavioral entropy from DS5 behavioral distributions.
  Characterise how uncertain or predictable each participant's behavior is across four primary
  information-theoretic dimensions (H_out, H_app, H_hour, H_emp) plus one StoP-derived bridge
  quantity (H_SDE). Compare entropy across RCT conditions. Explore psychosocial correlates.
- **Not goal:** Phenotype cluster assignment. Causal modelling of entropy mechanisms. Cross-dataset
  entropy comparison. Mutual information or transfer entropy (deferred to Step 2 — FLAG 4).
- **Output:** `outputs/model_outputs/information_theory/`

---

### FLAG decisions in effect

| Flag | Decision |
|---|---|
| FLAG 1 | Discrete Shannon entropy (paper Eq. 25) as primary estimator for H_out, H_app, H_hour. No kernel smoothing or bias correction. |
| FLAG 2 | Empirical PMF by raw frequency counts (no Laplace smoothing). H_emp uses K=10 uniform bins over [0, 1] for daily X_t series. |
| FLAG 3 | StoP bridge: `H_SDE = ½ log₂(2πe σ̂²)` — a Gaussian differential-entropy re-expression of the observed increment proxy `sig_hat`. It includes the binomial sampling floor and requires MIN_DAYS=7. |
| FLAG 4 | Mutual information I(app; outcome), conditional entropy H(outcome\|app), KL divergence from uniform, and transfer entropy are deferred to Step 2. |

### WARNING — proxy status of state variables

**Primary InT measures (H_out, H_app, H_hour, H_emp)** are derived from count-based behavioral
distributions. They quantify structural uncertainty in observed behavior. They do NOT directly
measure urge, self-control, habit strength, or any latent psychological state.

**H_SDE** additionally inherits the proxy limitation and sampling floor of the StoP state variable:
$X_t^{(i)} = n_{\text{opened,day}} / n_{\text{events,day}}$ is a count-based override rate,
not a directly observed psychological state. All entropy quantities derived from $X_t$ carry
this limitation.

This distinction must be maintained throughout interpretation. No entropy measure here
constitutes a direct measurement of behavioral uncertainty in a psychological sense.


---

## Section 1: Pure Theory

### 1.1 What Information Theory captures (paper §4.1.5)

Information Theory (Shannon, 1948) quantifies information and uncertainty in probabilistic terms.
Its core concept is **entropy** — a measure of how uncertain, diverse, or unpredictable a random
variable is. The less likely an event, the more information its occurrence provides.
Shannon entropy is defined for a discrete distribution over outcomes and is measured in bits
when the logarithm base is 2.

Information Theory addresses how information is represented, compressed, transmitted, and
transformed. At a theoretical level, it provides a language for asking how predictable a signal
is, how much two variables share, and whether one process reduces uncertainty about another.

Key branches relevant here:
- **Shannon entropy** — measures total uncertainty in a single distribution
- **Joint entropy** — measures total uncertainty in a joint distribution over two variables
- **Conditional entropy** — measures residual uncertainty in one variable given another is known
- **Mutual information** — measures shared information between two variables
- **KL divergence** — measures how much one distribution diverges from a reference distribution

Applied to smartphone behavior: entropy measures how predictable, diverse, or structured
a user's behavioral distribution is — across apps, across time, or across friction outcomes.
Unlike stochastic-process models (StoP), Information Theory does not model the *generative process*
of how behavior unfolds over time. Instead, it quantifies *distributional structure* in the
observed behavioral data.


### 1.2 Core equations from the paper (§4.1.5, Eq. 25–29)

**Shannon entropy** (paper **Eq. 25**) — the primary quantity:

$$H(X) = -\sum_{x \in \mathcal{X}} p(x) \log_2 p(x)$$

where $p(x)$ is the probability of outcome $x$. Entropy is high when many outcomes are plausible
and low when one outcome dominates. Units: bits (log base 2).

---

**Joint entropy** (paper **Eq. 26**):

$$H(X, Y) = -\sum_{x, y} p(x, y) \log_2 p(x, y)$$

Measures total uncertainty in the combined system of $X$ and $Y$.

---

**Conditional entropy** (paper **Eq. 27**):

$$H(Y | X) = H(X, Y) - H(X)$$

Uncertainty remaining in $Y$ once $X$ is known.

---

**Mutual information** (paper **Eq. 28**):

$$I(X; Y) = H(X) + H(Y) - H(X, Y)$$

How much knowing one variable reduces uncertainty about the other.
**Note:** Deferred to Step 2 in this notebook (FLAG 4).

---

**Kullback–Leibler divergence** (paper **Eq. 29**):

$$D_{KL}(P \| Q) = \sum_i p_i (\log p_i - \log q_i)$$

How much information is lost when $Q$ approximates $P$.
**Note:** Deferred to Step 2 in this notebook (FLAG 4).


### 1.3 Differential entropy of the Gaussian increment distribution (StoP bridge)

> **⚠ EQUATION GAP — This formula is NOT currently in the conceptual paper.**
> It is mathematically standard (Cover & Thomas, 2006, Ch. 8) and is used here as a bridge
> from the completed StoP notebook. It must be verified and added to the paper before submission
> (suggested location: §4.1.5 or §5.1 implementation section).

For a continuous Gaussian random variable $X \sim N(\mu, \sigma^2)$, the **differential entropy** is:

$$H_{\text{SDE}}^{(i)} = \frac{1}{2} \log_2\!\left(2\pi e\, \hat{\sigma}_i^2\right) \quad \text{(bits)}$$

**Derivation sketch:**
The StoP notebook estimates that the daily increment $\Delta X_n \sim N(\hat{\mu}, \hat{\sigma}^2)$.
The differential entropy of $N(\mu, \sigma^2)$ is $H = -\int p(x) \log p(x)\, dx = \frac{1}{2} \ln(2\pi e \sigma^2)$.
In bits: $H = \frac{1}{2} \log_2(2\pi e \sigma^2)$.

**Important properties of $H_{\text{SDE}}$:**
- $H_{\text{SDE}}$ is a **monotone increasing function** of $\hat{\sigma}$ — same rank order as `sig_hat`
- $H_{\text{SDE}}$ **can be negative** when $\hat{\sigma} < 1/\sqrt{2\pi e} \approx 0.242$ (valid for narrow distributions)
- It is used here as a **relative measure** only — comparison across participants and conditions
  is fully valid regardless of sign

**Role in this notebook:**
$H_{\text{SDE}}$ is the bridge between the StoP model and the Information Theory model.
It translates the diffusion parameter $\hat{\sigma}$ into the information-theoretic language of entropy.
Higher $\hat{\sigma}$ → higher $H_{\text{SDE}}$ → more uncertain / less predictable behavioral increments.


### 1.4 Information Theory applied to smartphone behavior (paper §4.2.5)

The paper (§4.2.5) frames InT application to smartphone behavior as follows:

- **Behavioral framing:** Entropy quantifies how predictable and diverse a user's behavioral
  repertoire is. A highly repetitive user who cycles through the same few apps has low entropy.
  A user with variable and unpredictable patterns has high entropy.

- **Intervention role:** Friction interventions may reduce or increase behavioral entropy —
  reducing it by channeling behavior toward fewer outcomes, or raising it by disrupting
  established habitual patterns.

- **Phenotype targets (deferred to Step 2):**
  - *Routine users* (low entropy): predictable, habitual, consistent behavior
  - *Exploratory users* (high entropy): variable, diverse, unpredictable behavior
  - *Context-coupled users*: behavior that is highly predictable conditional on context

The paper explicitly notes that InT does not model the *generative process* of how behavior
unfolds over time — that is the role of the StoP model. InT instead characterises the
*distributional structure* of observed behavior.

**Key caution for this notebook:**
Condition-level differences in entropy may reflect:
1. A genuine change in behavioral structure due to friction (interpretable)
2. Differences in the outcome distribution imposed by condition design (artifact)

In particular, H_out (outcome entropy) is directly affected by the different outcome-rate
distributions across conditions (control ~76% opened; friction conditions have different mixes).
This must be noted when interpreting condition comparisons of H_out.


### 1.5 StoP → InT unification connection (implemented here)

This connection is noted in `outputs/reports/stage4/stage4_implementation_plan.md` as:

> *StoP → InT: sig_hat characterises entropy of stationary distribution → connects diffusion
> to behavioral uncertainty*

The mathematical bridge: $H_{\text{SDE}}^{(i)} = \frac{1}{2} \log_2(2\pi e\, \hat{\sigma}_i^2)$

**What this bridge adds:**
- $H_{\text{SDE}}$ and $\hat{\sigma}$ carry identical **rank-order** information — they are monotone
  transforms of each other
- The **added value** of InT over StoP is the four independent entropy dimensions
  (H_out, H_app, H_hour, H_emp) that characterise behavioral structure StoP cannot see
- H_emp provides a **model-free** dispersion summary. $\rho(H_{\text{SDE}}, H_{\text{emp}})$ is a
  descriptive consistency check between increment-scale and level-scale dispersion — it is NOT a
  test of the Gaussian increment assumption embedded in StoP (see Section 8.1)

### 1.6 Unification connections (informational only)

| Connection | Mechanism | Status |
|---|---|---|
| StoP → InT | $H_{\text{SDE}} = \frac{1}{2}\log_2(2\pi e\hat{\sigma}^2)$ — implemented here | **Implemented (Section 5)** |
| InT → NeT | $H_{\text{app}}$ (behavioral repertoire entropy) complements network centralization measures | Noted in notebook 44 |
| InT → Phenotyping | (H_out, H_app, H_hour, H_emp, H_SDE) form candidate phenotype dimensions | Step 2, notebook 100 |
| InT → DyS | Low-entropy participants (low H_emp, low H_out) are candidates for deterministic DyS analysis | Noted in notebook 42 |
| InT Step 2 | Mutual information I(app; outcome), I(hour; outcome), transfer entropy | Deferred (FLAG 4) |


---

## Section 2: Final Equations Under FLAG Decisions

### State variables and distributions

| Symbol | Definition | Bounds | Source |
|---|---|---|---|
| $H_{\text{out}}^{(i)}$ | Shannon entropy (Eq. 25) over participant $i$'s outcome distribution $\{\text{opened, dismissed}\}$ | $[0,\ \log_2 3 \approx 1.585]$ bits | Computed from `event_outcome_simple` |
| $H_{\text{app}}^{(i)}$ | Shannon entropy (Eq. 25) over participant $i$'s app distribution | $[0,\ \log_2 K_{\text{apps}}]$ bits | Stage 1 feature `app_entropy` |
| $H_{\text{hour}}^{(i)}$ | Shannon entropy (Eq. 25) over participant $i$'s hour-of-day distribution | $[0,\ \log_2 24 \approx 4.585]$ bits | Stage 1 feature `hour_entropy` |
| $H_{\text{emp}}^{(i)}$ | Shannon entropy (Eq. 25) over participant $i$'s discretised daily $X_t$ series | $[0,\ \log_2 K]$ bits, $K=10$ | Computed from daily $X_t$ (same as StoP) |
| $H_{\text{SDE}}^{(i)}$ | Gaussian differential entropy: $\frac{1}{2}\log_2(2\pi e\hat{\sigma}_i^2)$ | $(-\infty, +\infty)$ bits | Derived from StoP `sig_hat` |

### Empirical PMF estimator (FLAG 1, FLAG 2)

$$\hat{p}(x) = \frac{\text{count}(x)}{n} \quad \text{(frequency-based, no smoothing)}$$

### H_emp discretisation (FLAG 2)

$$H_{\text{emp}}^{(i)} = -\sum_{k=1}^{K} \hat{p}_k \log_2 \hat{p}_k, \quad K = 10, \quad \text{bins: } [0.0, 0.1, \ldots, 1.0]$$

where $\hat{p}_k = \text{count}(X_t \in \text{bin}_k) / T_i$. Requires $T_i \geq 7$ (MIN_DAYS threshold).

### Theory-to-data mapping

| Theoretical quantity | Scientific meaning | DS5 source | Mapping | Confidence |
|---|---|---|---|---|
| $H_{\text{out}}$ | Uncertainty of friction-response outcome distribution | `event_outcome_simple` per participant | Direct (Eq. 25) | High |
| $H_{\text{app}}$ | Behavioral repertoire diversity across target apps | `app_entropy` (Stage 1) | Direct (Eq. 25) | High |
| $H_{\text{hour}}$ | Temporal context diversity (hour-of-day distribution) | `hour_entropy` (Stage 1) | Direct (Eq. 25) | High |
| $H_{\text{emp}}$ | Day-to-day unpredictability in override rate $X_t$ | Binned daily $X_t$ series | Derived (Eq. 25, FLAG 2) | Medium |
| $H_{\text{SDE}}$ | Uncertainty in Gaussian increment distribution (bridge) | StoP `sig_hat` | Derived (bridge, FLAG 3) | High (conditional on Gaussian assumption) |
| $p(\text{outcome})$ | Empirical outcome frequency | `event_outcome_simple` counts | Direct | High |
| Condition entropy shift | How RCT condition relates to behavioral entropy | `study_condition` from survey features | Derived | Medium |

### Equation gaps

| Equation | Status | Action |
|---|---|---|
| $H(X) = -\sum p(x)\log_2 p(x)$ (Eq. 25) | **In paper** | None |
| $H(X,Y) = -\sum p(x,y)\log_2 p(x,y)$ (Eq. 26) | In paper | Deferred to Step 2 |
| $H(Y\|X) = H(X,Y) - H(X)$ (Eq. 27) | In paper | Deferred to Step 2 |
| $I(X;Y) = H(X)+H(Y)-H(X,Y)$ (Eq. 28) | In paper | Deferred (FLAG 4) |
| $D_{KL}(P\|Q)$ (Eq. 29) | In paper | Deferred (FLAG 4) |
| $H_{\text{SDE}} = \frac{1}{2}\log_2(2\pi e\hat{\sigma}^2)$ | **NOT in paper** | **Add to paper before submission** |
| $\hat{p}(x) = \text{count}(x)/n$ | Implementation detail | No action needed |
| H_emp K-bin estimator | Implementation detail | Note in §5.1 of paper |


---

## Section 3: Setup and Data Loading

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

# ── Paths ──────────────────────────────────────────────────────────────────────
# Notebooks live in notebooks/ — go one level up to project root.
# nbconvert/nbclient sets the kernel cwd to the notebook's directory.
PROJECT_ROOT = Path('..').resolve()
CLEANED_DIR  = PROJECT_ROOT / 'outputs' / 'cleaned'
FEATURES_DIR = PROJECT_ROOT / 'outputs' / 'features' / 'ds5'
STOP_DIR     = PROJECT_ROOT / 'outputs' / 'model_outputs' / 'stochastic_processes'
OUTPUT_DIR   = PROJECT_ROOT / 'outputs' / 'model_outputs' / 'information_theory'
FIG_DIR      = OUTPUT_DIR / 'figures'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Parameters ─────────────────────────────────────────────────────────────────
MIN_DAYS       = 7          # minimum observation days — must match StoP (FLAG 2, FLAG 3)
K_BINS         = 10         # bins for H_emp discretisation (FLAG 2)
VALID_OUTCOMES = {'opened', 'dismissed'}
MIN_EVENTS_OUT = 10         # minimum events for H_out estimation (below: NaN)
PSYCH_CONDITIONS = {'default', 'reintervention'}  # friction-exposed only for psychosocial
PSYCH_MIN_N      = 20       # minimum n to report a psychosocial correlation

COND_ORDER  = ['control', 'default', 'reintervention']
COND_COLORS = {'control': '#4CAF50', 'default': '#2196F3', 'reintervention': '#FF9800'}
COND_LABELS = {'control': 'Control', 'default': 'Default', 'reintervention': 'Reintervention'}

print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Figures directory: {FIG_DIR.resolve()}")


In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────────
beh = pd.read_csv(
    CLEANED_DIR / 'ds5_behavioral_clean.csv',
    low_memory=False,
    parse_dates=['event_timestamp_utc']
)

stop_params = pd.read_csv(STOP_DIR / 'participant_parameters.csv')

surv = pd.read_csv(FEATURES_DIR / 'ds5_survey_features.csv')

bfeat = pd.read_csv(FEATURES_DIR / 'ds5_behavior_features.csv')

print(f"Behavioral events:          {len(beh):>8,}")
print(f"StoP eligible participants: {len(stop_params):>8,}")
print(f"Survey features:            {len(surv):>8,}")
print(f"Behavioral features:        {len(bfeat):>8,}")
print()
print("Outcome distribution (full behavioral file):")
print(beh['event_outcome_simple'].value_counts(dropna=False).to_string())


In [ ]:
# ── Filter and prepare ─────────────────────────────────────────────────────────
# Filter to valid friction outcomes (Eq. 25 estimator requires well-defined outcome set)
beh_valid = beh[beh['event_outcome_simple'].isin(VALID_OUTCOMES)].copy()

print(f"Valid friction events: {len(beh_valid):,} / {len(beh):,} "
      f"({len(beh_valid)/len(beh)*100:.1f}%)")
print(f"Participants with valid events: {beh_valid['participant_id_canonical'].nunique()}")

# Parse timestamp for date extraction
beh_valid['event_date'] = beh_valid['event_timestamp_utc'].dt.date

# DS5 structural check: study_condition is NOT in behavioral file — join from survey
cond_map = surv[['participant_id_canonical', 'study_condition']].drop_duplicates()
print()
print("study_condition distribution:")
print(cond_map['study_condition'].value_counts().to_string())


---

## Section 4: Primary Information Theory Measures (Eq. 25)

These four measures are estimated directly from DS5 behavioral distributions
using Shannon entropy (Eq. 25 from the paper). They do not depend on the StoP model.

**Measure overview:**

| Measure | Distribution | Source | Population |
|---|---|---|---|
| H_out | Outcome distribution {opened, dismissed} | Event-level behavioral file | All participants with ≥ MIN_EVENTS_OUT valid events |
| H_app | App distribution | Stage 1 `app_entropy` | All 573 participants |
| H_hour | Hour-of-day distribution | Stage 1 `hour_entropy` | All 573 participants |
| H_emp | Binned daily X_t distribution | Re-aggregated from behavioral file | MIN_DAYS ≥ 7 participants only |


### 4.1 Outcome entropy H_out

$H_{\text{out}}^{(i)} = -\sum_{o \in \{\text{opened, dismissed}\}} \hat{p}(o) \log_2 \hat{p}(o)$

Measures how uncertain or spread the friction-response distribution is per participant.  
Maximum value: $\log_2 2 = 1$ bit (uniform over the two decision outcomes).  
Minimum value: 0 bits (always same outcome).

**Interpretation caution:** H_out reflects both genuine behavioral uncertainty *and* the  
structural outcome mix imposed by experimental condition. Participants in control (majority  
`opened`) will tend to have lower H_out simply because their outcome distribution is more  
concentrated. Condition-level comparisons must account for this.

In [ ]:
def compute_shannon_entropy(series):
    """
    Compute Shannon entropy (bits, Eq. 25) from a categorical series.
    Returns NaN if fewer than MIN_EVENTS_OUT events.
    """
    if len(series) < MIN_EVENTS_OUT:
        return np.nan
    counts = series.value_counts()
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

h_out_df = (
    beh_valid
    .groupby('participant_id_canonical')['event_outcome_simple']
    .apply(compute_shannon_entropy)
    .reset_index()
    .rename(columns={'event_outcome_simple': 'H_out'})
)

print(f"H_out computed for {h_out_df['H_out'].notna().sum()} participants")
print(f"H_out NaN (< {MIN_EVENTS_OUT} valid events): {h_out_df['H_out'].isna().sum()}")
print()
print("H_out descriptive statistics (bits):")
print(h_out_df['H_out'].describe().round(4).to_string())
print(f"\nMax theoretical H_out (uniform over 2 outcomes): {np.log2(2):.4f} bits")


### 4.2 App entropy H_app and local-hour entropy H_hour

Both measures are recomputed here on the same opened/dismissed decision-event subset as H_out.  
This keeps the observation model aligned across the Information Theory measures.

- **H_app**: Shannon entropy over the distribution of target apps.  
  High value = behavior spread across many apps. Low value = dominated by one or two apps.  
  *Broader behavioral context — interpret more cautiously as a friction-response measure.*

- **H_hour**: Shannon entropy over the local hour-of-day distribution.  
  High value = phone use spread evenly across all hours. Low value = use concentrated at specific times.  
  *Temporal context structure — not directly related to friction response.*

In [ ]:
beh_valid['local_hour'] = pd.to_datetime(
    beh_valid['event_timestamp_local'], errors='coerce'
).dt.hour
h_app_df = (beh_valid.groupby('participant_id_canonical')['app_name']
            .apply(compute_shannon_entropy).rename('H_app').reset_index())
h_hour_df = (beh_valid.dropna(subset=['local_hour'])
             .groupby('participant_id_canonical')['local_hour']
             .apply(compute_shannon_entropy).rename('H_hour').reset_index())
h_app_hour_df = h_app_df.merge(h_hour_df, on='participant_id_canonical', how='outer')

print(f"H_app  available: {h_app_hour_df['H_app'].notna().sum()} participants")
print(f"H_hour available: {h_app_hour_df['H_hour'].notna().sum()} participants")
print()
print("H_app descriptive statistics (bits):")
print(h_app_hour_df['H_app'].describe().round(4).to_string())
print()
print("H_hour descriptive statistics (bits):")
print(h_app_hour_df['H_hour'].describe().round(4).to_string())


### 4.3 Empirical entropy of daily X_t series (H_emp)

$$H_{\text{emp}}^{(i)} = -\sum_{k=1}^{K} \hat{p}_k \log_2 \hat{p}_k, \quad K = 10, \quad \text{bins: } [0.0, 0.1, \ldots, 1.0]$$

Measures the distributional uncertainty in participant $i$'s daily override rate series.  
Requires MIN_DAYS=7 (same eligibility threshold as StoP).  
Maximum value: $\log_2 10 \approx 3.322$ bits (uniform over all 10 bins).

**Proxy limitation:** H_emp inherits the X_t proxy limitation — it measures uncertainty  
in observed override rate, not in any latent psychological state. It also has estimation  
noise for participants with short series (7–14 days).

In [ ]:
# ── Recompute daily X_t (same pipeline as StoP) ────────────────────────────────
beh_valid['opened_flag'] = (beh_valid['event_outcome_simple'] == 'opened').astype(int)

daily = (
    beh_valid
    .groupby(['participant_id_canonical', 'event_date'])
    .agg(n_opened=('opened_flag', 'sum'), n_events=('opened_flag', 'count'))
    .reset_index()
)
daily['X_t'] = daily['n_opened'] / daily['n_events']

# Apply MIN_DAYS threshold (same as StoP)
n_days_per_p = daily.groupby('participant_id_canonical').size()
eligible_ids = set(n_days_per_p[n_days_per_p >= MIN_DAYS].index)
print(f"Eligible participants (>= {MIN_DAYS} days): {len(eligible_ids)} "
      f"(matches StoP: {len(stop_params)})")


In [ ]:
# ── Compute H_emp per participant ──────────────────────────────────────────────
BIN_EDGES = np.linspace(0.0, 1.0, K_BINS + 1)


def empirical_entropy(x_series, bin_edges=BIN_EDGES):
    """
    Shannon entropy (bits) of discretised X_t series (FLAG 2).
    Uses K=10 uniform bins over [0, 1].
    """
    counts, _ = np.histogram(x_series, bins=bin_edges)
    total = counts.sum()
    if total == 0:
        return np.nan
    p = counts / total
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


daily_eligible = daily[daily['participant_id_canonical'].isin(eligible_ids)]

h_emp_records = []
for pid, group in daily_eligible.groupby('participant_id_canonical'):
    h = empirical_entropy(group['X_t'].values)
    h_emp_records.append({'participant_id_canonical': pid, 'H_emp': h})

h_emp_df = pd.DataFrame(h_emp_records)

print(f"H_emp computed for {len(h_emp_df)} participants")
print()
print("H_emp descriptive statistics (bits, K=10 bins):")
print(h_emp_df['H_emp'].describe().round(4).to_string())
print(f"\nMax theoretical H_emp (K=10 uniform): {np.log2(K_BINS):.4f} bits")


---

## Section 5: StoP → InT Bridge — H_SDE (FLAG 3)

> **Bridge quantity — derived from the completed StoP notebook (`43_ds5_stochastic_processes.ipynb`)**
> **Equation status: mathematically standard, NOT yet in the conceptual paper.**

$$H_{\text{SDE}}^{(i)} = \frac{1}{2} \log_2\!\left(2\pi e\, \hat{\sigma}_i^2\right) \quad \text{(bits)}$$

This is the **differential entropy** of the Gaussian increment distribution
$\Delta X_n \sim N(\hat{\mu}, \hat{\sigma}^2)$ as estimated in the StoP notebook.

**What this quantity is:**
- A monotone transform of StoP's `sig_hat` — same rank order, different units
- A bridge that places the StoP diffusion parameter within the information-theoretic framework
- A way to directly compare StoP-derived behavioral uncertainty with empirically estimated H_emp

**What this quantity is NOT:**
- An independent estimate from the behavioral data (it depends entirely on `sig_hat`)
- A replacement for the primary InT measures H_out, H_app, H_hour, H_emp
- A confirmed paper equation (must be added before submission)

**Note on negative values:** $H_{\text{SDE}} < 0$ when $\hat{\sigma} < 1/\sqrt{2\pi e} \approx 0.242$.
This is mathematically valid — differential entropy has no non-negativity constraint.
All participants are still fully comparable because the formula is monotone in $\hat{\sigma}$.


In [ ]:
# ── Compute H_SDE from StoP sig_hat (FLAG 3) ──────────────────────────────────
# H_SDE = ½ log₂(2πe σ̂²)
# This is a re-expression of the observed increment scale, including its
# finite-event binomial sampling floor; it is not a fitted Gaussian model.
# Source: Cover & Thomas (2006), Elements of Information Theory, Ch. 8
# ** NOT YET IN CONCEPTUAL PAPER — add before submission **

stop_params = stop_params.copy()
stop_params['H_SDE'] = 0.5 * np.log2(2.0 * np.pi * np.e * stop_params['sig_hat']**2)

print("H_SDE = ½ log₂(2πe σ̂²)  [observed-increment Gaussian proxy, bits]")
print("Source: StoP participant_parameters.csv (sig_hat)")
print()
print("H_SDE descriptive statistics (bits):")
print(stop_params['H_SDE'].describe().round(4).to_string())
print()
n_neg = (stop_params['H_SDE'] < 0).sum()
print(f"H_SDE < 0 (σ̂ < 0.242): {n_neg} / {len(stop_params)} participants ({n_neg/len(stop_params)*100:.1f}%)")
print()
print("Condition-level H_SDE means:")
print(stop_params.groupby('study_condition')['H_SDE'].mean().round(4).to_string())
print()
print("Note: these match the StoP sig_hat ordering exactly (monotone transform).")
print("  StoP sig_hat means: control=0.172, default=0.245, reintervention=0.200")


---

## Section 6: Participant Entropy Profile Assembly

All five entropy measures plus psychosocial variables are merged into a single  
per-participant profile. Coverage differs by measure:  
- H_out, H_app, H_hour: up to all 573 participants (those with sufficient events)  
- H_emp, H_SDE: MIN_DAYS ≥ 7 only (414 participants)

In [ ]:
# ── Assemble full entropy profile ─────────────────────────────────────────────
psych_cols = ['participant_id_canonical', 'study_condition',
              'fomo_mean_pre', 'belong_mean_pre', 'ostracism_mean_pre',
              'digital_wellbeing_pre', 'social_media_wellbeing_pre']

profile = (
    surv[psych_cols]
    .merge(h_out_df,       on='participant_id_canonical', how='left')
    .merge(h_app_hour_df,  on='participant_id_canonical', how='left')
    .merge(h_emp_df,       on='participant_id_canonical', how='left')
    .merge(
        stop_params[['participant_id_canonical', 'H_SDE', 'sig_hat', 'n_days']],
        on='participant_id_canonical', how='left'
    )
)

print("Entropy profile assembled.")
print(f"Total participants in profile: {len(profile)}")
print()
print("Measure availability (non-NaN):")
for col in ['H_out', 'H_app', 'H_hour', 'H_emp', 'H_SDE']:
    n = profile[col].notna().sum()
    print(f"  {col:<8}: {n:>4} / {len(profile)}")
print()
print("Condition distribution:")
print(profile['study_condition'].value_counts().to_string())


---

## Section 7: Condition-Level Entropy Comparison

For each entropy measure, we compare distributions across the three RCT conditions.  

**Interpretation notes:**  
- **H_SDE and H_emp**: condition ordering mirrors StoP sigma results (default > reintervention > control).  
  Since H_SDE = f(σ̂), this finding is a direct re-expression of the StoP result, not new evidence.  
- **H_out**: condition comparison is partially confounded by structural outcome-rate differences  
  across conditions. Report descriptively; do not overstate causal interpretation.  
- **H_app and H_hour**: these reflect broader behavioral patterns, not friction-response structure.  
  Condition differences here would be unexpected and require careful investigation.

In [ ]:
# ── Condition-level summary table ─────────────────────────────────────────────
entropy_measures = ['H_out', 'H_app', 'H_hour', 'H_emp', 'H_SDE']
measure_labels = {
    'H_out':  'H_out (outcome entropy, bits)',
    'H_app':  'H_app (app entropy, bits)',
    'H_hour': 'H_hour (hour entropy, bits)',
    'H_emp':  'H_emp (empirical X_t entropy, bits)',
    'H_SDE':  'H_SDE (StoP bridge, bits)',
}

rows = []
for measure in entropy_measures:
    for cond in COND_ORDER:
        sub = profile[
            (profile['study_condition'] == cond) & profile[measure].notna()
        ][measure]
        if len(sub) >= 3:
            rows.append({
                'measure': measure,
                'condition': cond,
                'n': int(len(sub)),
                'mean': round(sub.mean(), 4),
                'median': round(sub.median(), 4),
                'std': round(sub.std(), 4),
                'q25': round(sub.quantile(0.25), 4),
                'q75': round(sub.quantile(0.75), 4),
            })

cond_summary_df = pd.DataFrame(rows)
print("Condition-level entropy summary:")
print(cond_summary_df.to_string(index=False))


In [ ]:
# ── Figure 1: Entropy measures by condition ───────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(20, 6))
fig.suptitle('DS5 Information Theory — Entropy Distributions by Condition\n'
             '(Step 1 only: no phenotype labels)', fontsize=13, y=1.02)

measure_info = [
    ('H_out',  'H_out\n(outcome entropy, bits)',   f'Max = {np.log2(2):.3f} bits'),
    ('H_app',  'H_app\n(app entropy, bits)',        'Decision events'),
    ('H_hour', 'H_hour\n(local-hour entropy, bits)', 'Decision events'),
    ('H_emp',  'H_emp\n(X_t empirical, bits)',      f'Min-days ≥ {MIN_DAYS}'),
    ('H_SDE',  'H_SDE\n(StoP bridge, bits)',        '⚠ Bridge quantity\n(not in paper)'),
]

for ax, (measure, ylabel, note) in zip(axes, measure_info):
    parts = ax.violinplot(
        [profile[(profile['study_condition'] == c) & profile[measure].notna()][measure].values
         for c in COND_ORDER],
        positions=range(len(COND_ORDER)),
        showmedians=True,
        showmeans=False,
    )
    for i, (pc, cond) in enumerate(zip(parts['bodies'], COND_ORDER)):
        pc.set_facecolor(COND_COLORS[cond])
        pc.set_alpha(0.6)
    for key in ['cmedians', 'cmins', 'cmaxes', 'cbars']:
        parts[key].set_color('black')
        parts[key].set_linewidth(1.2)

    ax.set_xticks(range(len(COND_ORDER)))
    ax.set_xticklabels([COND_LABELS[c] for c in COND_ORDER], fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(note, fontsize=8, color='gray')

    # Overlay condition means
    for i, cond in enumerate(COND_ORDER):
        vals = profile[(profile['study_condition'] == cond) & profile[measure].notna()][measure]
        ax.scatter(i, vals.mean(), color='red', zorder=5, s=40, marker='D')

    if measure == 'H_SDE':
        ax.axhline(0, color='gray', lw=0.8, ls='--', alpha=0.5)

    ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig1_int_entropy_by_condition.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("Saved: fig1_int_entropy_by_condition.png")
print("Red diamond = condition mean. Horizontal dashed line in H_SDE panel = zero.")


---

## Section 8: StoP Bridge Validation and Entropy Dimension Correlations

**8.1 Bridge check:** If the Gaussian increment assumption from StoP is reasonable,  
$H_{\text{SDE}}$ and $H_{\text{emp}}$ should be correlated — they both estimate behavioral uncertainty  
from the same X_t series, but via different methods.  
$\rho(H_{\text{SDE}}, H_{\text{emp}})$ measures only whether increment-scale and level-scale dispersion co-vary. It does NOT test the Gaussian increment assumption: $H_{\text{SDE}}$ is a deterministic monotone function of $\hat{\sigma}$ and carries no distributional-shape information, and $H_{\text{emp}}$ is the entropy of daily levels, not of increments. Treat this as a descriptive consistency check only.

**8.2 Dimension correlations:** Are H_out, H_app, H_hour, H_emp, H_SDE all correlated  
(one behavioral complexity factor) or are they partially orthogonal dimensions?

In [ ]:
# ── 8.1 StoP bridge check: H_SDE vs H_emp ─────────────────────────────────────
bridge_df = profile[['participant_id_canonical', 'study_condition',
                      'sig_hat', 'H_SDE', 'H_emp']].dropna()
print(f"Bridge check participants (H_SDE and H_emp both available): {len(bridge_df)}")

r_bridge, p_bridge = stats.pearsonr(bridge_df['H_SDE'], bridge_df['H_emp'])
print(f"\nPearson r(H_SDE, H_emp) = {r_bridge:.4f}, p = {p_bridge:.4f}")
print("Interpretation:")
if abs(r_bridge) >= 0.7:
    print("  Strong co-variation of increment-scale and level-scale dispersion (descriptive only)")
elif abs(r_bridge) >= 0.4:
    print("  Moderate co-variation of increment-scale and level-scale dispersion (descriptive only)")
else:
    print("  Weak co-variation of increment-scale and level-scale dispersion (descriptive only)")
print("  NOTE: this is NOT a test of the Gaussian increment assumption — H_SDE is a")
print("  deterministic monotone transform of sig_hat and carries no shape information.")


In [ ]:
# ── Figure 2: H_SDE vs H_emp bridge check ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

for cond in COND_ORDER:
    sub = bridge_df[bridge_df['study_condition'] == cond]
    ax.scatter(sub['H_SDE'], sub['H_emp'],
               c=COND_COLORS[cond], label=COND_LABELS[cond],
               alpha=0.5, s=20, zorder=3)

# y = x reference line
lims = [min(bridge_df['H_SDE'].min(), bridge_df['H_emp'].min()),
        max(bridge_df['H_SDE'].max(), bridge_df['H_emp'].max())]
ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.4, label='y = x')

ax.set_xlabel('H_SDE (StoP bridge, bits)', fontsize=11)
ax.set_ylabel('H_emp (empirical X_t entropy, bits)', fontsize=11)
ax.set_title(f'StoP → InT Bridge Validation\n'
             f'Pearson r = {r_bridge:.3f} (p = {p_bridge:.3f}), n = {len(bridge_df)}\n'
             f'⚠ H_SDE equation not yet in paper (FLAG 3)', fontsize=10)
ax.legend(fontsize=9)
ax.axvline(0, color='gray', lw=0.8, ls=':', alpha=0.5)
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig2_int_stop_bridge.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("Saved: fig2_int_stop_bridge.png")


In [ ]:
# ── 8.2 Entropy dimension correlation matrix ───────────────────────────────────
corr_cols = ['H_out', 'H_app', 'H_hour', 'H_emp', 'H_SDE']
corr_data = profile[corr_cols].dropna()
print(f"Participants with all 5 entropy measures: {len(corr_data)}")

corr_matrix = corr_data.corr(method='pearson')
print("\nPearson correlation matrix:")
print(corr_matrix.round(3).to_string())


In [ ]:
# ── Figure 3: Entropy dimension correlation heatmap ───────────────────────────
import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=(7, 6))

labels = ['H_out', 'H_app', 'H_hour', 'H_emp', 'H_SDE*']
mat = corr_matrix.values

im = ax.imshow(mat, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=11)
ax.set_yticklabels(labels, fontsize=11)

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{mat[i, j]:.2f}', ha='center', va='center',
                fontsize=10, color='white' if abs(mat[i, j]) > 0.6 else 'black')

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Entropy Dimension Correlations (Pearson r)\n'
             f'n = {len(corr_data)} (all 5 measures available)\n'
             '* H_SDE = StoP bridge quantity', fontsize=10)

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig3_int_entropy_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("Saved: fig3_int_entropy_correlation_matrix.png")


---

## Section 9: Psychosocial Correlates of Entropy

**Scope:** Friction-exposed participants only (default + reintervention conditions).  
Consistent with the StoP psychosocial analysis (same subset, n ≈ 277).  

**Status:** Exploratory. Multiple comparisons across 5 entropy measures × 5 psychosocial scales  
= 25 tests. Report Pearson r and p-value. Flag as exploratory.  

**Cross-reference with StoP:** The StoP finding was:  
- FoMO ~ μ̂: r = −0.12, p = 0.047  
- FoMO ~ σ̂: r = −0.13, p = 0.036  

Since H_SDE = ½ log₂(2πe σ̂²) is a monotone transform of σ̂, the correlation  
ρ(FoMO, H_SDE) should be identical (up to rounding) to ρ(FoMO, σ̂).  
This serves as an internal consistency check.

In [ ]:
# ── Psychosocial correlates of entropy ────────────────────────────────────────
psych_vars = ['fomo_mean_pre', 'belong_mean_pre', 'ostracism_mean_pre',
              'digital_wellbeing_pre', 'social_media_wellbeing_pre']

psych_labels = {
    'fomo_mean_pre':               'FoMO (pre)',
    'belong_mean_pre':             'Belonging (pre)',
    'ostracism_mean_pre':          'Ostracism (pre)',
    'digital_wellbeing_pre':       'Digital wellbeing (pre)',
    'social_media_wellbeing_pre':  'Social media wellbeing (pre)',
}

# Restrict to friction-exposed conditions
psych_profile = profile[profile['study_condition'].isin(PSYCH_CONDITIONS)].copy()
print(f"Friction-exposed participants for psychosocial analysis: {len(psych_profile)}")

psych_results = []
for entropy_var in ['H_out', 'H_app', 'H_hour', 'H_emp', 'H_SDE']:
    for psych_var in psych_vars:
        sub = psych_profile[[entropy_var, psych_var]].dropna()
        if len(sub) >= PSYCH_MIN_N:
            r, p = stats.pearsonr(sub[entropy_var], sub[psych_var])
            psych_results.append({
                'entropy_var': entropy_var,
                'psych_var': psych_var,
                'n': int(len(sub)),
                'pearson_r': round(r, 4),
                'p_value': round(p, 4),
                'significant_p05': p < 0.05,
            })

psych_corr_df = pd.DataFrame(psych_results)

print("\nPsychosocial correlations with entropy measures (friction-exposed only):")
print(psych_corr_df.to_string(index=False))
print()
sig = psych_corr_df[psych_corr_df['significant_p05']]
print(f"Nominally significant (p < 0.05): {len(sig)} / {len(psych_corr_df)}")
if len(sig) > 0:
    print(sig.to_string(index=False))


In [ ]:
# ── Internal consistency check: FoMO ~ H_SDE vs FoMO ~ sig_hat ─────────────
fomo_hsde = psych_corr_df[
    (psych_corr_df['entropy_var'] == 'H_SDE') &
    (psych_corr_df['psych_var'] == 'fomo_mean_pre')
]
if len(fomo_hsde) > 0:
    r_hsde = fomo_hsde['pearson_r'].values[0]
    print(f"FoMO ~ H_SDE: r = {r_hsde:.4f}")
    print(f"StoP reference: FoMO ~ sig_hat: r = -0.1254, p = 0.036")
    print("These should match (monotone transform) — serves as pipeline consistency check.")


In [ ]:
# ── Figure 4: FoMO vs H_SDE and H_out ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Psychosocial Correlates of Entropy — Friction-Exposed Conditions Only\n'
             '(Exploratory, Step 1)', fontsize=11)

for ax, (entropy_var, title_note) in zip(axes, [
    ('H_SDE', '⚠ Bridge quantity (eq. not in paper)'),
    ('H_out',  'Direct InT measure (Eq. 25)'),
]):
    sub = psych_profile[['fomo_mean_pre', entropy_var, 'study_condition']].dropna()
    if len(sub) == 0:
        ax.set_visible(False)
        continue

    for cond in PSYCH_CONDITIONS:
        s = sub[sub['study_condition'] == cond]
        ax.scatter(s['fomo_mean_pre'], s[entropy_var],
                   c=COND_COLORS[cond], label=COND_LABELS[cond],
                   alpha=0.5, s=20)

    # Regression line
    slope, intercept, r, p, _ = stats.linregress(sub['fomo_mean_pre'], sub[entropy_var])
    x_line = np.linspace(sub['fomo_mean_pre'].min(), sub['fomo_mean_pre'].max(), 100)
    ax.plot(x_line, intercept + slope * x_line, 'k-', lw=1.5)

    ax.set_xlabel('FoMO pre-intervention', fontsize=11)
    ax.set_ylabel(f'{entropy_var} (bits)', fontsize=11)
    ax.set_title(f'{entropy_var} ~ FoMO\nr = {r:.3f}, p = {p:.3f}, n = {len(sub)}\n{title_note}',
                 fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig4_int_fomo_vs_entropy.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("Saved: fig4_int_fomo_vs_entropy.png")


---

## Section 10: StoP ↔ InT Comparison

### What StoP captures that InT does NOT

| StoP capability | Why InT misses it |
|---|---|
| **Drift direction μ̂** — average directional tendency | Entropy is symmetric; cannot distinguish drifting up vs down |
| **Temporal trajectory** — how $X_t$ evolves from day to day | InT treats the distribution as static; ignores the ordering of observations |
| **Friction effect as drift reduction h_c** — signed condition-level mechanistic parameter | InT's condition entropy comparison is not signed in the same sense |
| **Forward simulation** via Euler-Maruyama | InT provides no generative model for trajectories |
| **σ̂/\|μ̂\| impulsivity ratio** — relative dominance of noise vs drift | A single entropy value cannot decompose into drift and noise components |

### What InT captures that StoP does NOT

| InT capability | Why StoP misses it |
|---|---|
| **Behavioral repertoire diversity H_app** | StoP collapses all apps into the scalar $X_t$ |
| **Temporal context diversity H_hour** | StoP aggregates to daily resolution and ignores within-day time structure |
| **Outcome entropy H_out** — unpredictability of friction response | StoP uses $X_t$ as a continuous state; does not capture outcome-level distributional structure |
| **Model-free summary** — H_emp, H_out, H_app, H_hour require no Gaussian assumption | StoP requires Gaussian increment distribution (FLAG 1) |
| **Multi-dimensional entropy profile** — 5 distinct dimensions | StoP produces one 2D parameter space (μ̂, σ̂) |

### The bridge

- $H_{\text{SDE}} = \frac{1}{2}\log_2(2\pi e\hat{\sigma}^2)$ is a monotone transform of $\hat{\sigma}$
- The Pearson correlation $\rho(H_{\text{SDE}}, H_{\text{emp}})$ is a descriptive consistency check between increment-scale and level-scale dispersion — it does NOT test whether the Gaussian assumption is supported (see Section 8.1)
- The four primary InT dimensions (H_out, H_app, H_hour, H_emp) add information
  that **cannot be recovered from (μ̂, σ̂) alone**

### Step 2 unification vector

At Step 2, the per-participant phenotype vector would include:

$$\mathbf{v}^{(i)} = (\hat{\mu}_i,\ \hat{\sigma}_i,\ H_{\text{out}}^{(i)},\ H_{\text{app}}^{(i)},\ H_{\text{hour}}^{(i)},\ H_{\text{emp}}^{(i)})$$

This 6-dimensional vector spans process structure (StoP) and distributional uncertainty (InT).
$H_{\text{SDE}}$ is redundant with $\hat{\sigma}$ and is not needed in the phenotype vector.


In [ ]:
# ── Entropy dimension summary for StoP ↔ InT comparison ────────────────────────
print("=" * 65)
print("StoP ↔ InT COMPARISON SUMMARY")
print("=" * 65)

print("\n--- Confirmed StoP results (from notebook 43) ---")
print("  Drift μ̂:    control=0.007, default=0.007, reintervention=0.008")
print("  Diffusion σ̂: control=0.172, default=0.245, reintervention=0.200")
print("  FoMO ~ σ̂: r=-0.13, p=0.036 (friction-exposed, n=277)")

print("\n--- InT results (this notebook) ---")
for measure in ['H_out', 'H_app', 'H_hour', 'H_emp', 'H_SDE']:
    row = cond_summary_df[cond_summary_df['measure'] == measure]
    if len(row) > 0:
        ctrl = row[row['condition']=='control']['mean'].values
        dft  = row[row['condition']=='default']['mean'].values
        rein = row[row['condition']=='reintervention']['mean'].values
        if len(ctrl) and len(dft) and len(rein):
            tag = ' (bridge)' if measure == 'H_SDE' else ''
            print(f"  {measure}{tag}: control={ctrl[0]:.3f}, default={dft[0]:.3f}, reintervention={rein[0]:.3f}")

print(f"\n--- Bridge validation ---")
print(f"  ρ(H_SDE, H_emp) = {r_bridge:.3f} (n={len(bridge_df)})")


---

## Section 11: Interpretation and Limits

### What can be stated (Step 1)

1. **Behavioral entropy is estimable from DS5 data** across five distinct dimensions,
   each reflecting a different aspect of behavioral structure.

2. **H_SDE mirrors σ̂ from StoP** (by construction). The condition ordering (default > reintervention > control)
   reflects higher day-to-day behavioral variability in friction conditions — already documented in StoP.
   Reporting H_SDE as an entropy measure adds interpretive language, not new empirical content.

3. **H_emp vs H_SDE association** is a descriptive comparison of level-distribution and
   increment-scale dispersion. It does not test the Gaussian increment assumption.

4. **H_out by condition** shows how predictable friction responses are. Interpret descriptively.
   Do not overstate causal interpretation given the structural outcome-rate confound.

5. **H_app and H_hour** describe broader behavioral patterns. Condition differences would be surprising
   (friction does not change which apps participants use in general). Report as contextual descriptives.

6. **Psychosocial correlates** are exploratory (n ≈ 277, 25 tests). FoMO ~ H_SDE should replicate
   the StoP result by construction. Any additional significant correlations involving H_out, H_app,
   or H_hour would be genuinely new findings — report cautiously.

### What cannot be stated (Step 1)

- **No phenotype labels.** Entropy distributions are continuous dimensions. Do not assign
  "routine user" or "exploratory user" labels at this stage.
- **No causal claims within conditions.** Higher entropy in friction conditions does not prove
  that friction causes entropy increases. Entropy differences may also reflect participant selection
  into conditions (balanced by randomisation, but confounded by attrition and engagement).
- **No cross-dataset comparison.** These entropy estimates are DS5-specific and not yet
  harmonised with DS4, DS6, or DS1.

### Known limitations per measure

| Measure | Key limitation |
|---|---|
| H_out | Condition-confounded: different base outcome-rate distributions across conditions |
| H_SDE | Bridge equation not in paper; Gaussian increment assumption inherited from StoP FLAG 1 |
| H_emp | K=10 bin choice is arbitrary (FLAG 2); noisy for participants with 7–14 day windows |
| H_app | Decision-event app repertoire; it does not represent all phone use |
| H_hour | Local hour is available, but decision events do not represent all phone use |


---

## Section 12: Forward Connections

| Connection | Mechanism | Target |
|---|---|---|
| InT → Phenotyping (Step 2) | (H_out, H_app, H_hour, H_emp) join (μ̂, σ̂) as phenotype dimension candidates | Notebook 100_phenotype_exploration.ipynb |
| InT → NeT (44) | H_app entropy is the distributional complement of network centralization (app_herfindahl) | Notebook 44_ds5_network_models.ipynb |
| InT → DyS (42) | Low H_emp participants are candidates for deterministic DyS analysis (σ → 0 limit) | Notebook 42_ds5_dynamical_systems.ipynb |
| InT Step 2 extension | Mutual information I(app; outcome): does app identity predict outcome? (FLAG 4) | Step 2 |
| InT Step 2 extension | KL divergence from uniform outcome distribution: how "peaked" is friction response? | Step 2 |
| InT Step 2 extension | Conditional entropy H(outcome \| app): residual outcome uncertainty given app known | Step 2 |
| Paper §5.1 | InT + StoP is the natural candidate pair for the exploratory combination section (paper §5.2) | Thesis §5 |

### Note on the bridge equation for the paper

Before finalising the thesis or any publication from this work, the following equation
should be explicitly added to the conceptual paper (suggested: §4.1.5 or §5.1):

$$H = \frac{1}{2} \log_2(2\pi e\, \sigma^2)$$

With attribution: Cover & Thomas (2006), *Elements of Information Theory*, Chapter 8.
A brief derivation note (differential entropy of the Gaussian) and a statement
connecting it to the StoP diffusion parameter should accompany it.


---

## Section 13: Export Outputs

In [ ]:
# ── 1. participant_entropy_metrics.csv ────────────────────────────────────────
export_cols = ['participant_id_canonical', 'study_condition', 'n_days',
               'H_out', 'H_app', 'H_hour', 'H_emp', 'H_SDE', 'sig_hat']
export_profile = profile[export_cols].copy()

export_profile.to_csv(OUTPUT_DIR / 'participant_entropy_metrics.csv', index=False)
print(f"Saved: participant_entropy_metrics.csv  ({len(export_profile)} rows)")


In [ ]:
# ── 2. condition_entropy_summary.csv ─────────────────────────────────────────
cond_summary_df.to_csv(OUTPUT_DIR / 'condition_entropy_summary.csv', index=False)
print(f"Saved: condition_entropy_summary.csv  ({len(cond_summary_df)} rows)")


In [ ]:
# ── 3. stop_int_comparison.csv ────────────────────────────────────────────────
stop_int_df = bridge_df[['participant_id_canonical', 'study_condition',
                          'sig_hat', 'H_SDE', 'H_emp']].copy()
stop_int_df['H_SDE_minus_H_emp'] = stop_int_df['H_SDE'] - stop_int_df['H_emp']

stop_int_df.to_csv(OUTPUT_DIR / 'stop_int_comparison.csv', index=False)
print(f"Saved: stop_int_comparison.csv  ({len(stop_int_df)} rows)")
print(f"  Pearson r(H_SDE, H_emp) = {r_bridge:.4f}")


In [ ]:
# ── 4. psychosocial_entropy_correlations.csv ─────────────────────────────────
psych_corr_df.to_csv(OUTPUT_DIR / 'psychosocial_entropy_correlations.csv', index=False)
print(f"Saved: psychosocial_entropy_correlations.csv  ({len(psych_corr_df)} rows)")


In [ ]:
# ── 5. model_run_summary.md ───────────────────────────────────────────────────
run_ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Pre-build condition strings (avoid nested f-strings in older Python)
h_out_by_cond = {
    c: round(profile[profile['study_condition']==c]['H_out'].mean(), 4)
    for c in COND_ORDER if profile[profile['study_condition']==c]['H_out'].notna().any()
}
h_sde_by_cond = {
    c: round(stop_params[stop_params['study_condition']==c]['H_SDE'].mean(), 4)
    for c in COND_ORDER
}

sig_rows = psych_corr_df[psych_corr_df['significant_p05']]
sig_lines = [
    "  " + str(r.entropy_var) + " ~ " + str(r.psych_var) +
    ": r=" + str(r.pearson_r) + ", p=" + str(r.p_value) + ", n=" + str(r.n)
    for _, r in sig_rows.iterrows()
]
sig_summary = chr(10).join(sig_lines) if sig_lines else "  None"

h_out_lines = chr(10).join("  " + c + ": " + str(v) for c, v in h_out_by_cond.items())
h_sde_lines = chr(10).join("  " + c + ": " + str(v) for c, v in h_sde_by_cond.items())

avail = {col: str(profile[col].notna().sum()) + " / " + str(len(profile))
         for col in ["H_out", "H_app", "H_hour", "H_emp", "H_SDE"]}

summary_lines = [
    "# DS5 InT -- Model Run Summary",
    "",
    "**Run timestamp:** " + run_ts,
    "**Notebook:** notebooks/45_ds5_information_theory.ipynb",
    "**Model:** Information Theory (Stage 4 Step 1)",
    "**Dataset:** DS5 -- German Adolescents RCT (N=573 total)",
    "",
    "---",
    "",
    "## Pipeline parameters",
    "",
    "| Parameter | Value |",
    "|---|---|",
    "| MIN_DAYS (for H_emp, H_SDE) | " + str(MIN_DAYS) + " |",
    "| K_BINS (H_emp discretisation) | " + str(K_BINS) + " |",
    "| MIN_EVENTS_OUT (H_out threshold) | " + str(MIN_EVENTS_OUT) + " |",
    "| Valid outcomes | opened, dismissed |",
    "",
    "## FLAG decisions in effect",
    "",
    "| Flag | Decision |",
    "|---|---|",
    "| FLAG 1 | Discrete Shannon entropy (Eq. 25) -- no smoothing |",
    "| FLAG 2 | Empirical PMF by frequency counts; K=10 bins for H_emp |",
    "| FLAG 3 | H_SDE = half log2(2*pi*e*sigma^2) -- bridge from StoP (NOT in paper) |",
    "| FLAG 4 | Mutual information, conditional entropy, KL divergence deferred |",
    "",
    "## Measure availability",
    "",
    "| Measure | N participants |",
    "|---|---|",
    "| H_out  | " + avail["H_out"]  + " |",
    "| H_app  | " + avail["H_app"]  + " |",
    "| H_hour | " + avail["H_hour"] + " |",
    "| H_emp  | " + avail["H_emp"]  + " |",
    "| H_SDE  | " + avail["H_SDE"]  + " |",
    "",
    "## Condition-level means",
    "",
    "### H_out (outcome entropy, bits)",
    h_out_lines,
    "Max theoretical: " + str(round(float(__import__("numpy").log2(2)), 4)) + " bits (uniform over 2 outcomes)",
    "",
    "### H_SDE (StoP bridge, bits)",
    h_sde_lines,
    "Note: Ordering mirrors StoP sig_hat (monotone transform). Not independent evidence.",
    "",
    "## StoP bridge validation",
    "",
    "Pearson r(H_SDE, H_emp) = " + str(round(r_bridge, 4)) + " (n = " + str(len(bridge_df)) + ")",
    "",
    "## Psychosocial correlates (friction-exposed)",
    "",
    "Nominally significant (p < 0.05):",
    sig_summary,
    "",
    "## Output files",
    "",
    "| File | Description |",
    "|---|---|",
    "| participant_entropy_metrics.csv | Per-participant entropy measures (5 dimensions + sig_hat) |",
    "| condition_entropy_summary.csv | Condition-level entropy statistics |",
    "| stop_int_comparison.csv | H_SDE vs H_emp bridge check per participant |",
    "| psychosocial_entropy_correlations.csv | Pearson r of entropy vs psychosocial scales |",
    "| figures/fig1_int_entropy_by_condition.png | Entropy by condition (violin plots) |",
    "| figures/fig2_int_stop_bridge.png | H_SDE vs H_emp scatter (bridge validation) |",
    "| figures/fig3_int_entropy_correlation_matrix.png | Correlation heatmap of 5 dimensions |",
    "| figures/fig4_int_fomo_vs_entropy.png | FoMO vs H_SDE and H_out scatter |",
    "",
    "## Proxy warning",
    "",
    "All entropy measures describe observed behavioral distributions, not latent states.",
    "H_SDE depends on StoP proxy: X_t = n_opened_day / n_events_day.",
    "Bridge equation NOT yet in paper. Add before submission:",
    "  H = half * log2(2*pi*e*sigma^2)  [Cover & Thomas 2006, Ch. 8]",
    "",
    "## Known limitations",
    "",
    "1. H_out is condition-confounded (different base outcome rates by condition)",
    "2. H_SDE bridge equation not in paper -- required before submission",
    "3. H_emp estimation noisy for participants with 7-14 day windows",
    "4. H_app and H_hour describe decision events, not all phone use",
    "5. No bias correction for empirical entropy estimation (FLAG 1 and 2)",
    "6. Psychosocial analysis exploratory -- 25 tests, no multiple-comparison correction",
    "",
    "## Forward connections",
    "",
    "- InT -> Phenotyping (Step 2, notebook 100): entropy profile joins StoP parameters",
    "- InT -> NeT (44): H_app entropy complements network centralization",
    "- InT -> DyS (42): low H_emp participants are sigma->0 candidates",
    "- Step 2: mutual information, KL divergence, conditional entropy",
]

summary_md = chr(10).join(summary_lines)

with open(OUTPUT_DIR / 'model_run_summary.md', 'w') as f:
    f.write(summary_md)
print("Saved: model_run_summary.md")


In [ ]:
# ── Companion .md files for CSV outputs (CLAUDE.md requirement) ───────────────

md_participant = """# participant_entropy_metrics.csv

## Contents
Per-participant behavioral entropy metrics from the DS5 Information Theory notebook.
One row per DS5 participant in the survey features file (N up to 572).

## Unit of observation
One row per participant.

## Column reference

| Column | Type | Description |
|---|---|---|
| participant_id_canonical | str | Canonical participant identifier |
| study_condition | str | RCT condition: control / default / reintervention |
| n_days | float | Number of observation days used for H_emp/H_SDE (NaN if < MIN_DAYS) |
| H_out | float | Shannon entropy (bits) over outcome distribution {opened, dismissed} |
| H_app | float | Shannon entropy (bits) over app distribution — Stage 1 feature |
| H_hour | float | Shannon entropy (bits) over hour-of-day distribution — Stage 1 feature |
| H_emp | float | Shannon entropy (bits) of K=10 binned daily X_t series (NaN if < 7 days) |
| H_SDE | float | Gaussian differential entropy (bits) = ½ log₂(2πe σ̂²); bridge from StoP |
| sig_hat | float | StoP estimated diffusion parameter (NaN if < 7 days) |

## Production
- Source: outputs/cleaned/ds5_behavioral_clean.csv, outputs/features/ds5/, outputs/model_outputs/stochastic_processes/
- Producing notebook: notebooks/45_ds5_information_theory.ipynb
- Stage: Stage 4, Step 1

## Cleaning steps applied
- Behavioral file filtered to valid outcomes: opened, dismissed
- H_emp and H_SDE require MIN_DAYS >= 7 (same threshold as StoP)
- H_out requires MIN_EVENTS_OUT >= 10 valid events

## Known limitations
- H_SDE bridge equation NOT yet in conceptual paper
- H_out is condition-confounded by outcome-rate structure
- All measures describe behavioral distributions, not latent psychological states

## Load example
```python
import pandas as pd
from pathlib import Path
df = pd.read_csv(Path('outputs/model_outputs/information_theory/participant_entropy_metrics.csv'))
```
"""

md_condition = """# condition_entropy_summary.csv

## Contents
Condition-level summary statistics for all five entropy measures.

## Unit of observation
One row per (entropy_measure × condition) combination (15 rows total).

## Column reference
| Column | Type | Description |
|---|---|---|
| measure | str | Entropy variable name (H_out / H_app / H_hour / H_emp / H_SDE) |
| condition | str | RCT condition: control / default / reintervention |
| n | int | Number of participants with non-NaN values |
| mean | float | Condition mean (bits) |
| median | float | Condition median (bits) |
| std | float | Condition standard deviation (bits) |
| q25 | float | 25th percentile (bits) |
| q75 | float | 75th percentile (bits) |

## Production
- Producing notebook: notebooks/45_ds5_information_theory.ipynb
"""

md_bridge = """# stop_int_comparison.csv

## Contents
Per-participant comparison between H_SDE (StoP bridge) and H_emp (empirical).
Limited to participants with MIN_DAYS >= 7 (same as StoP eligibility).

## Unit of observation
One row per StoP-eligible participant (N=414).

## Column reference
| Column | Type | Description |
|---|---|---|
| participant_id_canonical | str | Participant identifier |
| study_condition | str | RCT condition |
| sig_hat | float | StoP diffusion estimate |
| H_SDE | float | Gaussian differential entropy from sig_hat (bits) |
| H_emp | float | Empirical entropy of binned daily X_t (bits) |
| H_SDE_minus_H_emp | float | Agreement gap between bridge and empirical estimates |

## Notes
H_SDE and H_emp summarise different aspects of the same daily series.
Their Pearson correlation is descriptive and is not a Gaussianity test.

## Production
- Producing notebook: notebooks/45_ds5_information_theory.ipynb
"""

md_psych = """# psychosocial_entropy_correlations.csv

## Contents
Pearson correlations between entropy measures and pre-intervention psychosocial scales.
Restricted to friction-exposed conditions (default + reintervention) for consistency with StoP.

## Unit of observation
One row per (entropy_var × psych_var) pair (up to 25 rows).

## Column reference
| Column | Type | Description |
|---|---|---|
| entropy_var | str | Entropy measure name |
| psych_var | str | Psychosocial variable name |
| n | int | Number of participants with both measures non-NaN |
| pearson_r | float | Pearson correlation coefficient |
| p_value | float | Two-tailed p-value |
| significant_p05 | bool | Whether p < 0.05 (exploratory flag only) |

## Notes
All correlations are exploratory. 25 tests performed — no multiple-comparison correction applied.
FoMO ~ H_SDE should match FoMO ~ sig_hat from StoP (monotone transform consistency check).

## Production
- Producing notebook: notebooks/45_ds5_information_theory.ipynb
"""

for fname, content in [
    ('participant_entropy_metrics.md', md_participant),
    ('condition_entropy_summary.md', md_condition),
    ('stop_int_comparison.md', md_bridge),
    ('psychosocial_entropy_correlations.md', md_psych),
]:
    with open(OUTPUT_DIR / fname, 'w') as f:
        f.write(content)
    print(f"Saved: {fname}")


In [ ]:
# ── Final output verification ──────────────────────────────────────────────────
print("=" * 55)
print("INFORMATION THEORY NOTEBOOK — OUTPUT VERIFICATION")
print("=" * 55)

expected_files = [
    'participant_entropy_metrics.csv',
    'participant_entropy_metrics.md',
    'condition_entropy_summary.csv',
    'condition_entropy_summary.md',
    'stop_int_comparison.csv',
    'stop_int_comparison.md',
    'psychosocial_entropy_correlations.csv',
    'psychosocial_entropy_correlations.md',
    'model_run_summary.md',
    'figures/fig1_int_entropy_by_condition.png',
    'figures/fig2_int_stop_bridge.png',
    'figures/fig3_int_entropy_correlation_matrix.png',
    'figures/fig4_int_fomo_vs_entropy.png',
]

all_ok = True
for fname in expected_files:
    path = OUTPUT_DIR / fname
    status = "OK" if path.exists() else "MISSING"
    if status == "MISSING":
        all_ok = False
    size = f"{path.stat().st_size:>8,} bytes" if path.exists() else ""
    print(f"  [{status}] {fname}  {size}")

print()
if all_ok:
    print("All expected outputs present.")
else:
    print("WARNING: Some outputs missing.")

print()
print("--- Key results ---")
print(f"  Participants with full entropy profile (all 5 measures): {len(corr_data)}")
print(f"  H_SDE condition means (bits):")
for _, row in cond_summary_df[cond_summary_df['measure']=='H_SDE'].iterrows():
    print(f"    {row['condition']}: {row['mean']:.4f}")
print(f"  H_out condition means (bits):")
for _, row in cond_summary_df[cond_summary_df['measure']=='H_out'].iterrows():
    print(f"    {row['condition']}: {row['mean']:.4f}")
print(f"  Bridge: r(H_SDE, H_emp) = {r_bridge:.4f}")
print()
print("  Nominally significant psychosocial correlates (p < 0.05):")
for _, r in psych_corr_df[psych_corr_df['significant_p05']].iterrows():
    print(f"    {r.entropy_var} ~ {r.psych_var}: r={r.pearson_r}, p={r.p_value}, n={r.n}")
